# Run Match

In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [ ]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [ ]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 20,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
#baseDB    = "Traxsource"
#baseDB    = "MyMixTapez"
#baseDB    = "Genius"
#baseDB    = "MetalArchives"
baseDB    = "RateYourMusic"  # 3

minL = 1
maxL = 6

minA = 1
maxA = 30000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource", "Genius", "MyMixTapez", "MetalArchives"] # "LastFM", "Deezer"]
#compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

In [ ]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=True, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

# Match & Cross Match

In [ ]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

In [ ]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)

In [ ]:
pdbm.mergeMultiRows()

# Extra Match

In [ ]:
# 

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5, minName=3, maxName=5)

In [ ]:
pdbm.include("""
314357d551   | Discogs        12667                                     3   5    | VÍMANA                                 VIMANA
337e0cfd25   | Spotify        7fl7AzgmjCh3Y1L2BKG07l                    3   6    | VÍMANA                                 VIMANA
cb5ca4d744   | Discogs        965837                                    3   5    | ROADRAGE                                ROAD RAGE
893b3b7b8d   | Discogs        226901                                    3   7    | BANDANA                                 BANANA
516817c148   | Discogs        4401217                                   3   7    | M.O.B.                                  M.O.B
52800081c7   | Discogs        799809                                    3   7    | AMBER #2                                AMBER#2
8f1797a14d   | Discogs        2565282                                   3   8    | BËEHLER                                BEEHLER
de084c9d67   | Discogs        1661113                                   3   8    | SR. TCEE                                SR. T.CEE
f0bec5688e   | Spotify        0OxYPhgB60MMrR7NormZMW                    3   5    | RED DOG                                 REDDOG
dd571f3af2   | Spotify        4ZW0NiaMKK1r8yzr9ms0S6                    3   5    | V.I.C.                                  V.I.C
98ee9507f8   | Discogs        351086                                    3   8    | LIL PAT                                 LIL' PAT
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5, minName=4)

In [ ]:
pdbm.include("""
8c8f602c56   | Discogs        6096765                                   5   4    | LILLY GOODMAN                           LILLY GOODMAN
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4, minName=3)

In [ ]:
pdbm.include("""
bb32f67cbe   | Genius         453859                                    5   3    | EDDY WALLY                              EDDY WALLY
36d51199ea   | Spotify        68jFQKiOk63vHQSTub3oCQ                    5   2    | CELTIC SPIRIT                           CELTIC SPIRIT
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2, minName=5)

In [ ]:
pdbm.include("""
b04ddc1dfc   | Genius         1233675                                   5   1    | MARGIE JOSEPH                           MARGIE JOSEPH
f9f546c46e   | Genius         227567                                    5   1    | ANTHONY MOORE                           ANTHONY MOORE
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f
#a6da8c9ee9   | 25875                    Discogs        678236                                  4    ALTAR OF FLESH                                    ALTAR OF FLIES                                     | a6da8c9ee9

# New Matching Code

# New Single Matching Code

In [ ]:
names = smdb.baseIO.getAvailableNames()

In [ ]:
smdb = SingleMatchDB(baseDB="RateYourMusic", compareDB="Spotify", reqs=reqs)
smdb.match()
smdb.save()
del smdb


mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
scmdb = SingleCrossMatchDB(baseDB, mres, crossreqs, verbose=True)
scmdb.match()
scmdb.save()
del scmdb


In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()

In [ ]:
mmeDF[mmeDF["RateYourMusic"] == '106836']

In [ ]:
mmeDF[mmeDF["Spotify"] == '3lk3F4u5qqxq8zFjwNj5U1']

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.masterSingle()
#pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
1d2402a17a   | 1061693                  Spotify        49D8h67pxvvUNGOLKEGjkx                  4    OWAIN ARWEL HUGHES                                OWAIN ARWEL HUGHES                                 | 1d2402a17a
a86b2ef789   | 121809                   Spotify        6NSIW1uEq8JZmxEkHMF17c                  4    ANNA TOMOWA-SINTOW                                ANNA TOMOWA-SINTOW                                 | a86b2ef789
877d262f5e   | 142182                   Spotify        5DwQvVHPVspRvStEAN722N                  4    TAKÁCS QUARTET                                   TAKÁCS QUARTET                                    | 877d262f5e
a2f65f8447   | 412578                   Spotify        50skve7Y0al39yGqLuCMNu                  4    MAURICE ABRAVANEL                                 MAURICE ABRAVANEL                                  | a2f65f8447
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
9554709d72   | 405351                   Spotify        2mHCS8PPaV7cAmT3ew8qHY                  2    SAULIUS SONDECKIS                                 SAULIUS SONDECKIS                                  | 9554709d72
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
3178f6847b   | 337551                   Spotify        7N0fh2csz0eFkrE01LF1m3                  1    STRATOS PAGIOUMTZIS                               STRATOS PAGIOUMTZIS                                | 3178f6847b
842d333cee   | 375588                   Spotify        2LqWWIvCBaetjLStxk1XK6                  1    VAN AND SCHENCK                                   VAN & SCHENCK                                      | 842d333cee
60cc9bc61a   | 77193                    Spotify        6VeTIJi6Dlx8ywPfIwqALY                  1    ALBERT NICHOLAS                                   ALBERT NICHOLAS                                    | 60cc9bc61a
""")
pdbm.masterSingle()
pdbm.merge(allownew=False, verbose=False)